# 01. Обзор данных — M.Video Search / NER

Ноутбук описывает схему и базовые характеристики трёх источников:

1. `query_clicks.parquet` — клики по поисковой выдаче (~31M строк)
2. `sku_desc.parquet` — названия и описания SKU (~1.18M строк)
3. `skus.pkl` — YML-каталог (Yandex Market Language)

> Для скорости используем **семплы**; полные размеры датасетов фиксируем в статистике.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_utils import (
    apply_plot_style, save_fig, ensure_dirs, load_query_clicks, load_sku_desc,
    text_len, parquet_num_rows, parquet_schema_names, dataset_overview_stats,
    save_stats, QUERY_CLICKS_PATH, SKU_DESC_PATH, SKUS_PKL_PATH,
    MVIDEO_RED, DARK_SLATE, MUTED,
)
ensure_dirs()
apply_plot_style()
pd.set_option("display.max_colwidth", 80)
print("ROOT:", ROOT)


## Полные размеры файлов


In [ ]:
n_clicks = parquet_num_rows(QUERY_CLICKS_PATH)
n_desc = parquet_num_rows(SKU_DESC_PATH)
print(f"query_clicks.parquet: {n_clicks:,} строк")
print(f"sku_desc.parquet:     {n_desc:,} строк")
print(f"skus.pkl size:        {SKUS_PKL_PATH.stat().st_size / 1e9:.2f} GB")
print("\nСырые колонки query_clicks:")
for c in parquet_schema_names(QUERY_CLICKS_PATH):
    print(" -", c)
print("\nКолонки sku_desc:")
for c in parquet_schema_names(SKU_DESC_PATH):
    print(" -", c)


## Семпл query_clicks

Колонки `toValidUTF8(...)` переименовываются в `sku_name`, `sku_brand_name`, `query_text`.


In [ ]:
SAMPLE_N = 300_000
df = load_query_clicks(n=SAMPLE_N, seed=42, random=True)
print("Форма семпла:", df.shape)
print("\ndtypes:")
print(df.dtypes)
display(df.head(8))
display(df.describe(include="all").T)


## Пропуски и пустые строки


In [ ]:
nulls = df.isna().sum()
empty = (df.astype(str).eq("") | df.astype(str).eq("nan")).sum()
overview = pd.DataFrame({"nulls": nulls, "empty_str": empty, "null_or_empty_share": (nulls + empty) / len(df)})
display(overview)
fig, ax = plt.subplots(figsize=(9, 4.5))
share = overview["null_or_empty_share"].sort_values()
ax.barh(share.index, share.values, color=MVIDEO_RED)
ax.set_title("Доля пропусков / пустых значений (query_clicks, семпл)")
ax.set_xlabel("Доля")
save_fig(fig, "16_nulls_overview.png")
plt.show()


## Семпл sku_desc


In [ ]:
desc = load_sku_desc(n=200_000, seed=42, random=True)
print("Форма:", desc.shape)
print(desc.dtypes)
display(desc.head(5))
print("\nПропуски:")
print(desc.isna().sum())
print("\nПустые title:", (desc["title"].fillna("").eq("")).sum())
print("Пустые description:", (desc["description"].fillna("").eq("")).sum())


## Структура skus.pkl (YML)

Файл большой (~1.5 GB). Загрузка может занять несколько минут.


In [ ]:
import pickle
with open(SKUS_PKL_PATH, "rb") as f:
    skus = pickle.load(f)
print(type(skus))
if isinstance(skus, dict):
    print("Ключи:", list(skus.keys())[:30])
    cat = skus.get("yml_catalog", skus)
    if isinstance(cat, dict):
        print("yml_catalog keys:", list(cat.keys())[:30])
        shop = cat.get("shop", {})
        if isinstance(shop, dict):
            print("shop keys:", list(shop.keys())[:40])


## Ключевые статистики → artifacts


In [ ]:
stats = dataset_overview_stats(df)
stats["known_full_unique_queries_approx"] = 1_790_000
stats["known_full_unique_skus_approx"] = 332_000
stats["known_full_unique_brands_approx"] = 7150
stats["known_full_unique_subjects_approx"] = 4103
stats["sample_top_queries"] = df["query_text"].value_counts().head(10).to_dict()
stats["sample_top_brands"] = (
    df["sku_brand_name"].replace("", np.nan).dropna().value_counts().head(10).to_dict()
)
save_stats(stats)
stats
